In [ ]:
!pip install pyspark


Create Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Amazon Reviews Big Data Analysis") \
    .getOrCreate()

Load Dataset

In [ ]:
df = spark.read.csv("Amazon_Product_Reviews.csv", header=True, inferSchema=True)

Explore Data

In [ ]:
df.show(5)
df.printSchema()
df.count()

+---+----------+--------------+--------------------+--------------------+----------------------+-----+----------+--------------------+--------------------+
| Id| ProductId|        UserId|         ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|             Summary|                Text|
+---+----------+--------------+--------------------+--------------------+----------------------+-----+----------+--------------------+--------------------+
|  1|B001E4KFG0|A3SGXH7AUHU8GW|          delmartian|                   1|                     1|    5|1303862400|Good Quality Dog ...|I have bought sev...|
|  2|B00813GRG4|A1D87F6ZCVE5NK|              dll pa|                   0|                     0|    1|1346976000|   Not as Advertised|"Product arrived ...|
|  3|B000LQOCH0| ABXLMWJIXXAIN|"Natalia Corres "...|                   1|                     1|    4|1219017600|"""Delight"" says...|"This is a confec...|
|  4|B000UA0QIQ|A395BORC6FGVXV|                Karl|            

32069

Total Reviews

In [ ]:
print("Total Reviews:", df.count())

Total Reviews: 32069


Ratings Distribution

In [ ]:
df.groupBy("Score").count().orderBy("Score").show()

+------------------+-----+
|             Score|count|
+------------------+-----+
|            Dad"""|    1|
|     Music Fan..."|    2|
|             RN"""|    3|
|     and Kitten"""|    9|
| and book lover"""|    1|
|         friend"""|    1|
|        swimme..."|    1|
|                 0|   62|
|                 1| 2902|
|                10|    1|
|                14|    2|
|                16|    1|
|                17|    1|
|                 2| 1831|
|                 3| 2649|
|                 4| 4631|
|                47|    2|
|                 5|19961|
|                 6|    2|
|                 7|    2|
+------------------+-----+
only showing top 20 rows


Average Rating

In [ ]:
# Read CSV PROPERLY
df = spark.read.csv(
    "Amazon_Product_Reviews.csv",
    header=True,
    inferSchema=True,
    multiLine=True,      # IMPORTANT ✅
    quote='"',           # handle quotes ✅
    escape='"'           # fix broken quotes ✅
)

In [ ]:
#Check schema
df.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



In [ ]:
# If still string → clean it
from pyspark.sql.functions import col

df = df.withColumn("Score", col("Score").cast("int"))
df = df.filter(df.Score.isNotNull())

In [ ]:
# Ratings Distribution
from pyspark.sql.functions import avg

df.select(avg("Score")).show()

+-----------------+
|       avg(Score)|
+-----------------+
|4.165750927759347|
+-----------------+



Negative Reviews

In [ ]:
df.filter(df.Score <= 2).show(5)

+---+----------+--------------+--------------+--------------------+----------------------+-----+----------+--------------------+--------------------+
| Id| ProductId|        UserId|   ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|             Summary|                Text|
+---+----------+--------------+--------------+--------------------+----------------------+-----+----------+--------------------+--------------------+
|  2|B00813GRG4|A1D87F6ZCVE5NK|        dll pa|                   0|                     0|    1|1346976000|   Not as Advertised|Product arrived l...|
|  4|B000UA0QIQ|A395BORC6FGVXV|          Karl|                   3|                     3|    2|1307923200|      Cough Medicine|If you are lookin...|
| 13|B0009XLVG0| A327PCT23YH90|            LT|                   1|                     1|    1|1339545600|My Cats Are Not F...|My cats have been...|
| 17|B001GVISJM|A3KLWF6WQ5BNYO|Erica Neathery|                   0|                     0|    2|1348

Sentiment Column

In [ ]:
from pyspark.sql.functions import when

df = df.withColumn("sentiment",
                   when(df.Score >= 3, "Positive").otherwise("Negative"))

df.groupBy("sentiment").count().show()

+---------+------+
|sentiment| count|
+---------+------+
| Positive|148861|
| Negative| 25352|
+---------+------+



In [ ]:

# Average rating
df.selectExpr("avg(Score)").show()

+-----------------+
|       avg(Score)|
+-----------------+
|4.176354434905012|
+-----------------+



In [ ]:
# Top 5 products with most reviews
df.groupBy("ProductId") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(5)

+----------+-----+
| ProductId|count|
+----------+-----+
|B002QWHJOU|  632|
|B002QWP89S|  632|
|B0026RQTGE|  632|
|B002QWP8H0|  632|
|B003B3OOPA|  623|
+----------+-----+
only showing top 5 rows


In [ ]:
# Most common rating
df.groupBy("Score").count().orderBy("Score").show()

+-----+------+
|Score| count|
+-----+------+
|    1| 52268|
|    2| 29769|
|    3| 42640|
|    4| 80655|
|    5|363122|
+-----+------+

